# 06 — Statistiques du jeu du classifieur

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lrivals/SerreIA/blob/main/notebooks/06_stats_classifieur.ipynb)

Vue statistique de `data/index.parquet` (docs/19 §3) : effectifs et déséquilibre,
couverture de la taxonomie par source, cohérence des splits, géométrie, photométrie et
écart de domaine PlantVillage / CCMT vs PlantDoc, quasi-doublons et bruit d'étiquetage,
embeddings (optionnel). Le calcul est dans `vision/datasets/stats.py` ; les chiffres clés
et les figures vont dans `reports/datasets/classifieur/`. CPU seulement.

## Paramètres

In [ ]:
N_SAMPLE = 4000  # images de l'échantillon photométrique (stratifié source × classe)
MIN_VAL_TEST = 20  # alerte : classes sous ce seuil en val ou en test
CLF_RUN = None  # classifieur des embeddings (runs/<CLF_RUN>/best.pt) ; None : sautés
SEED = 0
DATASETS = ["index", "plantvillage", "ccmt", "plantdoc"]

## Environnement

In [ ]:
# Environnement (PC et Colab) : dépôt, données, sorties (docs/data-drive.md).
# PC : rien n'est installé ; un jeu absent est récupéré du Drive par rclone, sinon la cellule
# s'arrête avec la commande à lancer. Colab : clone du dépôt à REV, installation, données via
# le Drive monté (archives MyDrive/SerreIA/data/*.tar), sorties précédentes restaurées.
import logging
import os
import subprocess
import sys
from pathlib import Path

DRIVE_DIR = "/content/drive/MyDrive/SerreIA"  # Colab : data/*.tar et runs/ ; None : pas de Drive
REPO_URL = "https://github.com/lrivals/SerreIA.git"
REV = "main"
COLAB = "google.colab" in sys.modules

if COLAB:
    ROOT = Path("/content/SerreIA")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin", REV], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[vision,stats]"], check=True
    )
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)

from tools.notebooks import drive_data  # noqa: E402

if COLAB and DRIVE_DIR:
    drive_data.mount_drive()
    drive_data.restore_all(DRIVE_DIR)
drive_data.ensure(DATASETS, DRIVE_DIR if COLAB else None)

# Colab : tous les cœurs pour le chargement, base MLflow séparée de celle du PC.
ENV_SET = (
    [f"data.num_workers={os.cpu_count()}", f"mlflow.tracking_uri={drive_data.COLAB_MLFLOW}"]
    if COLAB
    else []
)

## Index

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from common.report import REPORTS_DIR, file_sha256, write_report
from vision.datasets import stats as st
from vision.datasets import stats_plots as sp
from vision.datasets.build_index import (
    DATA_DIR, INDEX_NAME, REJECTS_NAME, low_eval_classes, read_index_meta,
)
from vision.datasets.dataset import load_index

OUT = REPORTS_DIR / "classifieur"
OUT.mkdir(parents=True, exist_ok=True)
raw = pd.read_parquet(DATA_DIR / INDEX_NAME)  # avec les groupes contradictoires
df = load_index(DATA_DIR / INDEX_NAME)  # ce que voit l'entraînement
meta = read_index_meta(DATA_DIR / INDEX_NAME)
SUMMARY = {
    "taxonomy_version": meta["taxonomy_version"],
    "index_sha256": file_sha256(DATA_DIR / INDEX_NAME),
    "n_images_index": len(raw),
    "n_images": len(df),
    "ecartees_contradictoires": len(raw) - len(df),
    "n_especes": df["espece"].nunique(),
    "n_classes": df["classe"].nunique(),
}
SECTIONS = {}
display(pd.Series(SUMMARY).to_frame("valeur"))

## 3.1 Effectifs et déséquilibre

Tables après retrait des groupes contradictoires (ce que voit l'entraînement), puis
source × split avant retrait. Indicateurs sur train : ratio max/min, entropie
normalisée (1 : uniforme), Gini (0 : uniforme), nombre effectif de classes.

In [ ]:
for rows in ["source", "espece", "classe"]:
    print(f"{rows} × split")
    display(st.counts(df, rows))
print("source × split avant retrait des groupes contradictoires")
display(st.counts(raw, "source"))
SECTIONS["Source × split"] = st.counts(df, "source")

In [ ]:
levels = st.imbalance_by_level(df)
display(levels)
tail = st.long_tail(df)
display(sp.plot_long_tail(tail, OUT / "longue_traine.png"))
low = low_eval_classes(df, MIN_VAL_TEST)
print(f"classes sous {MIN_VAL_TEST} images en val ou en test : {len(low)}")
display(pd.Series(low, name="effectifs", dtype=str).to_frame())
SUMMARY.update(desequilibre_train=levels.to_dict(orient="index"), classes_sous_min=low)
SECTIONS["Déséquilibre (train)"] = levels

## 3.2 Couverture de la taxonomie

Les classes **mono-source** sont celles où le modèle peut apprendre la source (fond
uni PlantVillage…) plutôt que le symptôme : à rapporter à part en cross-domain.

In [ ]:
ct, share = st.coverage(df)
display(ct)
display(share.round(2))
mono = st.mono_source(df)
print(f"{len(mono)} classes mono-source sur {df['classe'].nunique()}")
display(mono)
amb = st.ambiguous_weight(df)
print(f"classes ambiguës : {len(amb['classes'])}, {amb['n_images']} images "
      f"({amb['part']:.1%})", amb["par_classe"])
masks = st.mask_share(df)
display(masks.to_frame())
SUMMARY.update(classes_mono_source=mono["source"].to_dict(), ambigues=amb,
               part_masque=masks.to_dict())
SECTIONS["Classes mono-source"] = mono

## 3.3 Cohérence des splits

Divergence de Jensen-Shannon (bits, 0 : distributions identiques) entre les
distributions de classes, χ² classe × split par source (p élevée attendue : splits
stratifiés), fuite de groupes (doit afficher 0), classes de `test_xdomain` absentes
du train.

In [ ]:
div = st.split_divergence(df)
display(pd.Series(div["js"], name="JS (bits)").to_frame())
display(pd.DataFrame(div["chi2"]).T)
leaks = st.group_leaks(df)
print(f"groupes dans plusieurs splits : {len(leaks)} (attendu : 0)")
gaps = st.xdomain_gaps(df)
print("test_xdomain sans train :", gaps["xdomain_sans_train"])
print("train sans test_xdomain :", len(gaps["train_sans_xdomain"]), "classes")
SUMMARY.update(js_splits=div["js"], chi2_par_source=div["chi2"],
               fuites_groupes=len(leaks), xdomain=gaps)

## 3.4 Géométrie

Part des images dont le petit côté est sous la résolution d'entrée (agrandies au
redimensionnement).

In [ ]:
geo = st.geometry(df, st.INPUT_SIZE)
display(geo)
display(sp.plot_distributions(df.assign(aspect=df["width"] / df["height"]),
                              ["width", "height", "aspect"], log=(),
                              path=OUT / "geometrie.png"))
SUMMARY["part_petites_images"] = geo[("petite", f"part<{st.INPUT_SIZE}")].to_dict()

## 3.5 Photométrie et écart de domaine

Sur `N_SAMPLE` images : luminosité, contraste, saturation, teinte dominante, netteté
(variance du laplacien), part de fond uni (bordure proche de sa médiane). L'écart
PlantDoc vs PV + CCMT explique la chute cross-domain (docs/14) ; `hors_plage_ref` est
la part de PlantDoc hors du [p1, p99] du domaine d'entraînement.

In [ ]:
photo = st.photometry_table(df, DATA_DIR, N_SAMPLE, SEED)
photo.to_csv(OUT / "photometrie.csv", index=False)
display(sp.plot_distributions(photo, list(st.DESCRIPTORS), path=OUT / "photometrie.png"))
display(sp.plot_pca(st.descriptor_pca(photo), path=OUT / "photometrie_pca.png"))
gap = st.domain_gap(photo)
display(gap)
SUMMARY["ecart_domaine"] = gap.to_dict(orient="index")
SECTIONS["Écart de domaine (PlantDoc vs PV + CCMT)"] = gap

In [ ]:
# Mêmes descripteurs après les augmentations fortes (docs/03 §3, sans remplacement de
# fond) : couvrent-elles la plage de PlantDoc ? hors_plage_ref : PlantDoc hors du batch.
import torch

from vision.datasets.augment import build_transform
from vision.datasets.batch_augment import BatchAugment
from vision.datasets.dataset import read_rgb
from vision.eval.gradcam import denormalize

torch.manual_seed(SEED)
labo = st.stratified_sample(df[df["source"].isin(["plantvillage", "ccmt"])], 256, SEED)
strong = build_transform("strong", st.INPUT_SIZE)
x = torch.stack([strong(image=read_rgb(DATA_DIR / p))["image"] for p in labo["path"]])
aug = st.photometry_images(denormalize(BatchAugment("strong")(x)), source="augmenté")
both = pd.concat([photo, aug], ignore_index=True)
display(sp.plot_distributions(both, list(st.DESCRIPTORS),
                              path=OUT / "photometrie_augmente.png"))
gap_aug = st.domain_gap(both, ref=["augmenté"])
display(gap_aug)
SUMMARY["ecart_plantdoc_vs_augmente"] = gap_aug.to_dict(orient="index")

## 3.6 Quasi-doublons et bruit d'étiquetage

Paires candidates (pHash ≤ 6) et leur distance dHash : la bimodalité justifie le seuil
de confirmation dHash ≤ 8 (docs/02 §3). Puis les groupes contradictoires (même image
dans plusieurs classes) et les fichiers rejetés.

In [ ]:
gs = st.group_sizes(raw)
print(f"{gs['n_groupes_doublons']} groupes de quasi-doublons, {gs['images_en_doublon']} "
      f"images, plus gros groupe : {gs['max']} (attendu : 21)")
display(pd.Series(gs["distribution"], name="groupes").rename_axis("taille").to_frame().T)
pairs = st.candidate_pair_distances(raw["phash"].to_numpy(), raw["dhash"].to_numpy(),
                                    meta["max_hamming"])
confirmed = (pairs["d_dhash"] <= meta["max_hamming_confirm"]).sum()
print(f"{len(pairs)} paires candidates (pHash ≤ {meta['max_hamming']}), "
      f"{confirmed} confirmées (dHash ≤ {meta['max_hamming_confirm']})")
display(sp.plot_hash_distances(pairs, meta["max_hamming_confirm"],
                               OUT / "distances_hash.png"))
SUMMARY.update(groupes=gs, paires_candidates=len(pairs), paires_confirmees=confirmed)

In [ ]:
contra = st.contradictory_groups(raw)
print(f"{contra['n_groupes']} groupes contradictoires, {contra['n_images']} images",
      contra["par_source"])
display(contra["paires"].head(20))
classe_of = raw.set_index("path")["classe"].astype(str)
ex = [p for ps in contra["groupes"]["paths"].head(8) for p in ps[:3]]
if ex:
    display(sp.plot_image_grid([DATA_DIR / p for p in ex],
                               [classe_of[p].split("/", 1)[1] for p in ex],
                               path=OUT / "contradictoires.png"))
rejects = st.read_rejects(DATA_DIR / REJECTS_NAME)
print(f"{len(rejects)} fichiers rejetés")
display(rejects.groupby(["source", "dossier"]).size().rename("n").to_frame())
SUMMARY.update(contradictoires={k: contra[k] for k in ["n_groupes", "n_images",
                                                       "par_source"]},
               rejets=rejects["source"].value_counts().to_dict())
SECTIONS["Paires de classes des groupes contradictoires"] = contra["paires"].head(20)

## 3.7 Embeddings (optionnel)

Features du classifieur `CLF_RUN` sur un échantillon, projetées en 2D (UMAP, sinon
t-SNE). Une séparation nette par source au sein d'une même espèce signale un
raccourci de domaine. Seule cellule qui profite d'un GPU.

In [ ]:
if CLF_RUN is None:
    print("CLF_RUN = None : embeddings sautés")
else:
    from vision.train.train_classifier import pick_device

    emb = st.stratified_sample(df, min(N_SAMPLE, 2000), SEED)
    feats = st.extract_features(ROOT / "runs" / CLF_RUN / "best.pt", emb["path"].tolist(),
                                DATA_DIR, str(pick_device(None)))
    xy, method = st.embed_2d(feats, seed=SEED)
    display(sp.plot_embedding(xy, emb["source"], f"{method} : source",
                              OUT / "embeddings_source.png"))
    display(sp.plot_embedding(xy, emb["espece"], f"{method} : espèce",
                              OUT / "embeddings_espece.png"))

## Rapport

In [ ]:
write_report(OUT, SUMMARY, SECTIONS)
drive_data.sync_all(DRIVE_DIR)
display(Markdown((OUT / "summary.md").read_text(encoding="utf-8")))

## Résumé

`reports/datasets/classifieur/` : `summary.json` (chiffres clés, version de la
taxonomie, hash de l'index), `summary.md`, `photometrie.csv` et les figures. À recouper
avec docs/02 : PlantVillage train 43 434, CCMT train 20 111, crops PlantDoc 8 704, plus
gros groupe 21, `pomme_de_terre/sain` à 15 en val et en test. Décisions (docs/19 §9) :
classes mono-source à suivre en cross-domain, augmentations au vu de l'écart de
domaine, nettoyage des groupes contradictoires CCMT.